In [4]:
import re
from pathlib import Path

import py3Dmol
import ipywidgets as widgets
from IPython.display import display

In [5]:
XYZ_FILE = Path("dump.xyz")

def index_xyz(filename):
    """
    Store only the byte offset, atom count, and timestep for every frame.
    Does NOT load coordinates into memory.
    """
    offsets = []
    timesteps = []
    natoms_list = []

    with open(filename, "r") as f:
        while True:
            offset = f.tell()
            line = f.readline()

            if not line:
                break
            if not line.strip():
                continue

            natoms = int(line.strip())
            comment = f.readline().strip()

            match = re.search(r"Timestep:\s*([+-]?\d+)", comment)
            timestep = int(match.group(1)) if match else len(offsets)

            offsets.append(offset)
            timesteps.append(timestep)
            natoms_list.append(natoms)

            # Skip atom lines without storing them
            for _ in range(natoms):
                f.readline()

    return offsets, timesteps, natoms_list


frame_offsets, timesteps, natoms_list = index_xyz(XYZ_FILE)

print(f"Frames: {len(frame_offsets)}")
print(f"Atoms/frame: {natoms_list[0]}")
print(f"First timestep: {timesteps[0]}")
print(f"Last timestep: {timesteps[-1]}")

Frames: 401
Atoms/frame: 1150
First timestep: 0
Last timestep: 20000


In [6]:
# Visualization-only mapping.
# Change these as desired.
TYPE_TO_ELEMENT = {
    "1": "C",
    "2": "O",
}

def read_frame(frame_index):
    """
    Read exactly one XYZ frame from disk.
    Returns an XYZ-format string understood by py3Dmol.
    """
    with open(XYZ_FILE, "r") as f:
        f.seek(frame_offsets[frame_index])

        natoms = int(f.readline().strip())
        comment = f.readline().strip()

        lines = [str(natoms), comment]

        for _ in range(natoms):
            fields = f.readline().split()

            atom_type = fields[0]
            x, y, z = fields[1:4]

            element = TYPE_TO_ELEMENT.get(atom_type, "C")
            lines.append(f"{element} {x} {y} {z}")

    return "\n".join(lines)

In [7]:
# ------------------------------------------------------------
# Cell 4: interactive MD viewer with play/pause + fixed bounds
# ------------------------------------------------------------

# First determine global trajectory bounds once.
# This scans the file but does NOT store the trajectory in memory.
def get_trajectory_bounds():
    xmin = ymin = zmin = float("inf")
    xmax = ymax = zmax = float("-inf")

    with open(XYZ_FILE, "r") as f:
        while True:
            line = f.readline()

            if not line:
                break

            if not line.strip():
                continue

            natoms = int(line.strip())

            # Skip comment/timestep line
            f.readline()

            for _ in range(natoms):
                fields = f.readline().split()

                x = float(fields[1])
                y = float(fields[2])
                z = float(fields[3])

                xmin = min(xmin, x)
                xmax = max(xmax, x)

                ymin = min(ymin, y)
                ymax = max(ymax, y)

                zmin = min(zmin, z)
                zmax = max(zmax, z)

    return xmin, xmax, ymin, ymax, zmin, zmax


# Calculate fixed bounds
xmin, xmax, ymin, ymax, zmin, zmax = get_trajectory_bounds()

# Add some visual padding around the system
padding = 2.0

xmin -= padding
xmax += padding
ymin -= padding
ymax += padding
zmin -= padding
zmax += padding

print("Fixed trajectory bounds:")
print(f"x: {xmin:.2f} to {xmax:.2f}")
print(f"y: {ymin:.2f} to {ymax:.2f}")
print(f"z: {zmin:.2f} to {zmax:.2f}")


# ------------------------------------------------------------
# Controls
# ------------------------------------------------------------

frame_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=len(frame_offsets) - 1,
    step=1,
    description="Frame:",
    continuous_update=False,
    layout=widgets.Layout(width="600px")
)

play = widgets.Play(
    value=0,
    min=0,
    max=len(frame_offsets) - 1,
    step=1,
    interval=50,       # milliseconds/frame
    description="Play"
)

# Loop continuously
play.repeat = True

# Synchronize Play widget with slider
widgets.jslink(
    (play, "value"),
    (frame_slider, "value")
)

info = widgets.HTML()
output = widgets.Output()


# ------------------------------------------------------------
# Display one frame
# ------------------------------------------------------------

def show_frame(i):

    info.value = (
        f"<b>Frame:</b> {i} &nbsp;&nbsp; "
        f"<b>Timestep:</b> {timesteps[i]}"
    )

    with output:

        # Remove previous displayed viewer
        output.clear_output(wait=True)

        view = py3Dmol.view(
            width=800,
            height=600
        )

        # ----------------------------------------------------
        # Model 0: real MD atoms
        # ----------------------------------------------------

        view.addModel(
            read_frame(i),
            "xyz"
        )

        view.setStyle(
            {"model": 0, "elem": "C"},
            {
                "sphere": {
                    "color": "gray",
                    "radius": 0.55
                }
            }
        )

        view.setStyle(
            {"model": 0, "elem": "O"},
            {
                "sphere": {
                    "color": "red",
                    "radius": 0.55
                }
            }
        )


        # ----------------------------------------------------
        # Model 1: invisible atoms defining fixed bounds
        # ----------------------------------------------------

        dummy_xyz = f"""2
fixed bounds
He {xmin} {ymin} {zmin}
He {xmax} {ymax} {zmax}
"""

        view.addModel(
            dummy_xyz,
            "xyz"
        )

        # No visible representation for dummy atoms
        view.setStyle(
            {"model": 1},
            {}
        )


        # ----------------------------------------------------
        # Viewer appearance
        # ----------------------------------------------------

        view.setBackgroundColor("white")

        # Because dummy atoms always occupy the same extrema,
        # the zoom/scale is identical for every frame.
        view.zoomTo()

        view.show()


# ------------------------------------------------------------
# Slider callback
# ------------------------------------------------------------

def slider_changed(change):

    if change["name"] == "value":
        show_frame(change["new"])


frame_slider.observe(
    slider_changed,
    names="value"
)


# ------------------------------------------------------------
# Display controls and initial frame
# ------------------------------------------------------------

controls = widgets.HBox([
    play,
    frame_slider
])

display(
    controls,
    info,
    output
)

show_frame(0)

Fixed trajectory bounds:
x: -28.04 to 28.04
y: -28.06 to 28.04
z: -28.05 to 28.06


HTML(value='')

Output()